
# Word Embeddings & Vector Databases for RAG

## Technologies Covered
- FAISS
- Pinecone
- ChromaDB
- Weaviate

## Learning Objectives
- Understand embeddings mathematically
- Learn how embeddings work in Retrieval Augmented Generation (RAG)
- Create embeddings using Sentence Transformers
- Store and retrieve embeddings using vector databases
- Perform semantic similarity search
- Hands-on exercises for participants

---

## Real World Context
Example business statements used in this notebook:
- "Ashi is working on a RAG project for document search."
- "The analytics team is building a chatbot using vector databases."
- "Pinecone helps store embeddings for semantic retrieval."



# 1. What are Word Embeddings?

Word embeddings convert text into numerical vectors.

Traditional NLP:
- "cat" → unique ID
- "dog" → unique ID

Problem:
- Machines cannot understand semantic meaning.

Embeddings solve this by placing similar words/sentences close together in vector space.

Example:
- "RAG system"
- "Retrieval augmented generation"

These embeddings will be mathematically close.

---

# Embedding Mathematics

A sentence embedding is represented as:

\[
E = [x_1, x_2, x_3, ..., x_n]
\]

Where:
- Each value represents a learned feature
- Typical dimensions:
  - 384
  - 768
  - 1536

Similarity is measured using:

## Cosine Similarity

\[
Similarity(A,B)=\frac{A.B}{||A|| ||B||}
\]

Where:
- \(A.B\) = dot product
- \(||A||\) = vector magnitude

Higher cosine similarity means semantically similar content.


In [1]:

# Install Required Libraries
# Uncomment if running first time

# !pip install sentence-transformers faiss-cpu chromadb weaviate-client pinecone-client pandas numpy


In [2]:

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully")


C:\Users\Suyashi144893\AppData\Local\anaconda3\envs\pytorch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully


In [3]:

# Load Embedding Model

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded")


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2615.95it/s]


Embedding model loaded



# Example 1 — Basic Sentence Embeddings for RAG

Goal:
Convert project-related business sentences into embeddings and compare semantic similarity.


In [4]:

sentences = [
    "Ashi is working on a RAG project.",
    "The team is building a retrieval augmented generation application.",
    "A data analyst is preparing dashboards in Power BI.",
    "FAISS is useful for vector similarity search."
]

embeddings = model.encode(sentences)

print("Embedding Shape:", embeddings.shape)


Embedding Shape: (4, 384)


In [5]:

# Cosine Similarity Matrix

similarity_matrix = cosine_similarity(embeddings)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=sentences,
    columns=sentences
)

similarity_df


,Ashi is working on a RAG project.,The team is building a retrieval augmented generation application.,A data analyst is preparing dashboards in Power BI.,FAISS is useful for vector similarity search.
Ashi is working on a RAG project.,1.000000,0.191027,0.181568,0.069515
The team is building a retrieval augmented generation application.,0.191027,1.000000,0.252849,0.265650
A data analyst is preparing dashboards in Power BI.,0.181568,0.252849,1.000000,0.130121
FAISS is useful for vector similarity search.,0.069515,0.265650,0.130121,1.000000



## Observation

You will notice:
- RAG-related sentences are more similar.
- Power BI sentence is semantically different.
- FAISS sentence has moderate similarity because it relates to retrieval/search.

This is the foundation of RAG retrieval systems.



# Example 2 — Using FAISS for Vector Search

FAISS (Facebook AI Similarity Search) is widely used for:
- Semantic search
- Document retrieval
- RAG pipelines
- Recommendation systems


In [6]:

import faiss

documents = [
    "Ashi is working on a RAG project using LangChain.",
    "Pinecone stores embeddings for semantic retrieval.",
    "ChromaDB is lightweight and developer friendly.",
    "Weaviate supports vector search with metadata filtering.",
    "Machine learning models require training data."
]

doc_embeddings = model.encode(documents)

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(doc_embeddings).astype('float32'))

print("Number of vectors stored:", index.ntotal)


Number of vectors stored: 5


In [11]:

query = "vector database for RAG"

query_embedding = model.encode([query])

k = 2

distances, indices = index.search(
    np.array(query_embedding).astype('float32'),
    k
)

print("Query:", query)

print("\nTop Matches:")

for i in indices[0]:
    print("-", documents[i])


Query: vector database for RAG

Top Matches:
- Weaviate supports vector search with metadata filtering.
- Ashi is working on a RAG project using LangChain.



# Understanding Vector Databases

## 1. FAISS
- Local vector search library
- Fast similarity search
- Good for prototypes and local systems

## 2. Pinecone
- Managed cloud vector database
- Scalable production deployment
- Popular in enterprise RAG systems

## 3. ChromaDB
- Open-source lightweight vector database
- Easy integration with LangChain

## 4. Weaviate
- Graph + vector database
- Supports hybrid search
- Metadata filtering

---

# RAG Pipeline Flow

1. User asks question
2. Convert query into embedding
3. Search vector database
4. Retrieve relevant chunks
5. Send context to LLM
6. Generate grounded answer


In [8]:

# Pinecone Sample Structure

'''
from pinecone import Pinecone

pc = Pinecone(api_key="YOUR_API_KEY")

index = pc.Index("rag-demo")

index.upsert([
    ("1", embeddings[0].tolist(), {"text":"Ashi is working on a RAG project"})
])

results = index.query(
    vector=query_embedding[0].tolist(),
    top_k=2,
    include_metadata=True
)

print(results)
'''


'\nfrom pinecone import Pinecone\n\npc = Pinecone(api_key="YOUR_API_KEY")\n\nindex = pc.Index("rag-demo")\n\nindex.upsert([\n    ("1", embeddings[0].tolist(), {"text":"Ashi is working on a RAG project"})\n])\n\nresults = index.query(\n    vector=query_embedding[0].tolist(),\n    top_k=2,\n    include_metadata=True\n)\n\nprint(results)\n'

In [9]:

# ChromaDB Example

'''
import chromadb

client = chromadb.Client()

collection = client.create_collection(name="rag_collection")

collection.add(
    documents=[
        "Ashi is working on a RAG project",
        "FAISS enables semantic retrieval"
    ],
    ids=["1", "2"]
)

results = collection.query(
    query_texts=["vector search in RAG"],
    n_results=2
)

print(results)
'''


'\nimport chromadb\n\nclient = chromadb.Client()\n\ncollection = client.create_collection(name="rag_collection")\n\ncollection.add(\n    documents=[\n        "Ashi is working on a RAG project",\n        "FAISS enables semantic retrieval"\n    ],\n    ids=["1", "2"]\n)\n\nresults = collection.query(\n    query_texts=["vector search in RAG"],\n    n_results=2\n)\n\nprint(results)\n'

In [10]:

# Weaviate Example

'''
import weaviate

client = weaviate.Client("http://localhost:8080")

# Example object insertion
client.data_object.create(
    {
        "text": "Ashi is working on a RAG project"
    },
    class_name="RAGDocs"
)

# Semantic search
response = client.query.get(
    "RAGDocs",
    ["text"]
).with_near_text({
    "concepts": ["vector retrieval"]
}).do()

print(response)
'''


'\nimport weaviate\n\nclient = weaviate.Client("http://localhost:8080")\n\n# Example object insertion\nclient.data_object.create(\n    {\n        "text": "Ashi is working on a RAG project"\n    },\n    class_name="RAGDocs"\n)\n\n# Semantic search\nresponse = client.query.get(\n    "RAGDocs",\n    ["text"]\n).with_near_text({\n    "concepts": ["vector retrieval"]\n}).do()\n\nprint(response)\n'


# Hands-On Exercise 1

## Task
Create embeddings for the following:
- "The HR team is building an AI assistant."
- "The finance team uses Power BI dashboards."
- "RAG applications improve enterprise search."

## Questions
1. Which sentences are semantically closer?
2. Why?
3. What happens if you change the query sentence?

---

# Hands-On Exercise 2

## Task
Create a FAISS index using:
- 10 custom business sentences

Then:
1. Search using:
   - "semantic search"
   - "AI chatbot"
2. Retrieve top 3 results
3. Observe similarity behavior

---

# Additional Practice

Try replacing:
- FAISS → ChromaDB
- ChromaDB → Pinecone
- Pinecone → Weaviate

Observe:
- Ease of setup
- Retrieval speed
- Scalability



# RAG Architecture Summary

## Retrieval Augmented Generation Components

### Step 1 — Chunking
Split documents into smaller chunks.

### Step 2 — Embedding Generation
Convert chunks into vectors.

### Step 3 — Store in Vector DB
Use:
- FAISS
- Pinecone
- ChromaDB
- Weaviate

### Step 4 — Semantic Retrieval
Find most relevant chunks.

### Step 5 — LLM Response Generation
Provide retrieved context to LLM.

---

# Key Learning

Embeddings are the backbone of modern RAG systems.



# Conclusion

You learned:
- What embeddings are
- Mathematical intuition behind embeddings
- Cosine similarity
- Semantic search
- FAISS implementation
- Pinecone workflow
- ChromaDB workflow
- Weaviate workflow
- RAG retrieval architecture
